# 13 — カメラキャリブレーション

## 目標
- `cv2.findChessboardCorners` でコーナーを検出
- `cv2.calibrateCamera` で内部パラメータと歪み係数を推定
- `cv2.undistort` で歪み補正

## モデル
カメラ行列 K:
```
K = [[fx, 0, cx],
     [0, fy, cy],
     [0,  0,  1]]
```
歪み係数: `(k1, k2, p1, p2, k3)`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import show_nb
from main import generate_synthetic_chessboard, calibrate

COLS, ROWS = 9, 6
images = generate_synthetic_chessboard(COLS, ROWS, square=60, n=15)
print(f'合成チェスボード生成: {len(images)} 枚')

In [ ]:
# コーナー検出の可視化
pairs = []
for i, img in enumerate(images[:4]):
    gray = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    color = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    ret, corners = cv2.findChessboardCorners(gray, (COLS, ROWS), None)
    if ret:
        cv2.drawChessboardCorners(color, (COLS, ROWS), corners, ret)
    pairs.append((f'corners {i} (found={ret})', color))
show_nb(pairs, cols=2)

In [ ]:
K, dist, rms = calibrate(images, COLS, ROWS, square_m=0.025)
if K is not None:
    print(f'\nRMS: {rms:.4f} px')
    print(f'fx={K[0,0]:.1f}, fy={K[1,1]:.1f}, cx={K[0,2]:.1f}, cy={K[1,2]:.1f}')
    print(f'dist: {dist.ravel()}')

In [ ]:
if K is not None:
    sample = images[0] if images[0].ndim == 3 else cv2.cvtColor(images[0], cv2.COLOR_GRAY2BGR)
    h, w = sample.shape[:2]
    K_new, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1)
    undist = cv2.undistort(sample, K, dist, None, K_new)
    show_nb([('distorted', sample), ('undistorted', undist)], cols=2)